In [5]:
import sys, math
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import norm

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_DIR))
sys.path.insert(0, str(PROJECT_DIR / 'scripts'))

from qne.cascade.key import key_from_sifted_json
from qne.cascade.finite_key import finite_key_output_length, asymptotic_key_length, cascade_leakage, h, v
from randextract import ToeplitzHashing

print("Setup complete")

Setup complete


In [2]:
alice_key, alice_indices = key_from_sifted_json(
    str(PROJECT_DIR / "results" / "fabric_alice_sifted_bits.json"), "alice_bits")
bob_key, bob_indices = key_from_sifted_json(
    str(PROJECT_DIR / "results" / "fabric_bob_sifted_bits.json"), "bob_bits")

assert alice_indices == bob_indices
real_qber = alice_key.nr_bits_different(bob_key) / alice_key.get_nr_bits()
print(f"Loaded {alice_key.get_nr_bits()} bits, real_qber = {real_qber:.4f}")

Loaded 3876 bits, real_qber = 0.0114


In [3]:
key_pairs_df = pd.read_csv(str(PROJECT_DIR / "results" / "key_pairs_index.csv"))
print(key_pairs_df)

   index  n_bits      qber                                         alice_path  \
0      0    3810  0.011024  /home/fabric/work/qkd-dependability/results/al...   
1      1    3794  0.009489  /home/fabric/work/qkd-dependability/results/al...   
2      2    3761  0.013826  /home/fabric/work/qkd-dependability/results/al...   
3      3    3815  0.009699  /home/fabric/work/qkd-dependability/results/al...   
4      4    3890  0.010540  /home/fabric/work/qkd-dependability/results/al...   

                                            bob_path  
0  /home/fabric/work/qkd-dependability/results/bo...  
1  /home/fabric/work/qkd-dependability/results/bo...  
2  /home/fabric/work/qkd-dependability/results/bo...  
3  /home/fabric/work/qkd-dependability/results/bo...  
4  /home/fabric/work/qkd-dependability/results/bo...  


In [4]:
from qne.cascade.finite_key import finite_key_output_length, asymptotic_key_length
from randextract import ToeplitzHashing

n = alice_key.get_nr_bits()
Q = real_qber

out_len_finite, r, t = finite_key_output_length(n, Q)
out_len_asymptotic = asymptotic_key_length(n, Q)
out_len_placeholder = ToeplitzHashing.calculate_length(
    extractor_type="quantum", input_length=n,
    relative_source_entropy=0.5, error_bound=1e-6,
)

print(f"n = {n} bits, Q (real QBER) = {Q:.4f}")
print(f"Cascade leakage r = {r:.1f} bits, verification hash t = {t:.1f} bits")
print()
print(f"Asymptotic (Shor-Preskill):   {out_len_asymptotic} bits")
print(f"Placeholder (0.5 entropy):    {out_len_placeholder} bits")
print(f"Finite-key corrected:         {out_len_finite} bits")

n = 3876 bits, Q (real QBER) = 0.0114
Cascade leakage r = 428.3 bits, verification hash t = 11.9 bits

Asymptotic (Shor-Preskill):   3181 bits
Placeholder (0.5 entropy):    1900 bits
Finite-key corrected:         2282 bits


In [10]:
"""
Apply the validated finite-key formula to your real, collected key pairs,
comparing finite-key-corrected length against the asymptotic estimate
and the placeholder ToeplitzHashing calculation currently used elsewhere
in the pipeline.
"""
import pandas as pd
from qne.cascade.key import key_from_sifted_json
from qne.cascade.finite_key import finite_key_output_length, asymptotic_key_length
from randextract import ToeplitzHashing

# --- Run across all 5 collected key pairs ---
key_files = [(f"alice_sifted_bits_key{i}.json", f"bob_sifted_bits_key{i}.json") for i in range(5)]

rows = []
for key_idx, (a_file, b_file) in enumerate(key_files):
    a_path = PROJECT_DIR / "results" / a_file
    b_path = PROJECT_DIR / "results" / b_file
    if not a_path.exists() or not b_path.exists():
        print(f"key{key_idx}: files not found, skipping")
        continue

    a_key, a_idx = key_from_sifted_json(str(a_path), "alice_bits")
    b_key, b_idx = key_from_sifted_json(str(b_path), "bob_bits")
    assert a_idx == b_idx, f"key{key_idx}: indices don't match!"

    n = a_key.get_nr_bits()
    Q = a_key.nr_bits_different(b_key) / n

    ell_finite, r, t = finite_key_output_length(n, Q)
    ell_asymptotic = asymptotic_key_length(n, Q)
    ell_placeholder = ToeplitzHashing.calculate_length(
        extractor_type="quantum", input_length=n,
        relative_source_entropy=0.5, error_bound=1e-6,
    )

    reduction_pct = (1 - ell_finite / ell_asymptotic) * 100 if ell_asymptotic > 0 else float('nan')

    rows.append({
        "key_index": key_idx, "n_bits": n, "qber": Q,
        "cascade_leakage_r": r, "verification_hash_t": t,
        "ell_asymptotic": ell_asymptotic,
        "ell_placeholder": ell_placeholder,
        "ell_finite_key": ell_finite,
        "reduction_vs_asymptotic_pct": reduction_pct,
    })
    print(f"key{key_idx}: n={n}, Q={Q:.4f} -> "
          f"asymptotic={ell_asymptotic}, placeholder={ell_placeholder}, "
          f"finite-key={ell_finite} ({reduction_pct:.1f}% reduction)")

# --- Also include your original, first-validated key (outside the collect_key_pairs set) ---
orig_a = PROJECT_DIR / "results" / "fabric_alice_sifted_bits.json"
orig_b = PROJECT_DIR / "results" / "fabric_bob_sifted_bits.json"
if orig_a.exists() and orig_b.exists():
    a_key, a_idx = key_from_sifted_json(str(orig_a), "alice_bits")
    b_key, b_idx = key_from_sifted_json(str(orig_b), "bob_bits")
    if a_idx == b_idx:
        n = a_key.get_nr_bits()
        Q = a_key.nr_bits_different(b_key) / n
        ell_finite, r, t = finite_key_output_length(n, Q)
        ell_asymptotic = asymptotic_key_length(n, Q)
        ell_placeholder = ToeplitzHashing.calculate_length(
            extractor_type="quantum", input_length=n,
            relative_source_entropy=0.5, error_bound=1e-6,
        )
        reduction_pct = (1 - ell_finite / ell_asymptotic) * 100 if ell_asymptotic > 0 else float('nan')
        rows.append({
            "key_index": "original", "n_bits": n, "qber": Q,
            "cascade_leakage_r": r, "verification_hash_t": t,
            "ell_asymptotic": ell_asymptotic,
            "ell_placeholder": ell_placeholder,
            "ell_finite_key": ell_finite,
            "reduction_vs_asymptotic_pct": reduction_pct,
        })
        print(f"original: n={n}, Q={Q:.4f} -> "
              f"asymptotic={ell_asymptotic}, placeholder={ell_placeholder}, "
              f"finite-key={ell_finite} ({reduction_pct:.1f}% reduction)")

df_finitekey = pd.DataFrame(rows)
print("\n=== Summary across all real keys ===")
print(df_finitekey.to_string(index=False))

df_finitekey.to_csv(str(PROJECT_DIR / "results" / "finite_key_analysis_all_keys.csv"), index=False)
print(f"\nSaved -> {PROJECT_DIR / 'results' / 'finite_key_analysis_all_keys.csv'}")

print(f"\nMean reduction vs. asymptotic: {df_finitekey['reduction_vs_asymptotic_pct'].mean():.1f}%")

key0: n=3810, Q=0.0110 -> asymptotic=3143, placeholder=1867, finite-key=2251 (28.4% reduction)
key1: n=3794, Q=0.0095 -> asymptotic=3206, placeholder=1859, finite-key=2311 (27.9% reduction)
key2: n=3761, Q=0.0138 -> asymptotic=2969, placeholder=1842, finite-key=2092 (29.5% reduction)
key3: n=3815, Q=0.0097 -> asymptotic=3213, placeholder=1869, finite-key=2316 (27.9% reduction)
key4: n=3890, Q=0.0105 -> asymptotic=3233, placeholder=1907, finite-key=2330 (27.9% reduction)
original: n=3876, Q=0.0114 -> asymptotic=3181, placeholder=1900, finite-key=2282 (28.3% reduction)

=== Summary across all real keys ===
key_index  n_bits     qber  cascade_leakage_r  verification_hash_t  ell_asymptotic  ell_placeholder  ell_finite_key  reduction_vs_asymptotic_pct
        0    3810 0.011024         412.071285            11.895575            3143             1867            2251                    28.380528
        1    3794 0.009489         364.455669            11.889504            3206             185

In [5]:
"""
Compare SDC fault-injection mismatch rates when using the finite-key-
corrected output length vs. the (currently-used) asymptotic/placeholder
output length. Uses key0's real data, Mock session, same fault
probabilities as your earlier sweeps.
"""
import numpy as np
from galois import GF2
from randextract import ToeplitzHashing
from qne.cascade.finite_key import finite_key_output_length, asymptotic_key_length
from qne.cascade.fault_injection import SDCFaultInjector


def run_with_custom_length(alice_key, bob_reconciled, out_len, toeplitz_prob, hash_prob, seed):
    """Run Toeplitz extraction + fault injection with a MANUALLY SPECIFIED
    output length, instead of the placeholder-based calculate_length call."""
    n_bits = alice_key.get_nr_bits()
    injector = SDCFaultInjector(toeplitz_matrix_prob=toeplitz_prob,
                                  hash_output_prob=hash_prob, seed=seed + 2)

    ext = ToeplitzHashing(input_length=n_bits, output_length=out_len)
    pa_seed = GF2.Random(ext.seed_length)

    alice_final = injector.fast_toeplitz_extract_with_fault(ext, GF2(alice_key.bits), pa_seed, "alice")
    alice_final = injector.maybe_flip_hash_output_bit(alice_final, "alice")
    bob_final = injector.fast_toeplitz_extract_with_fault(ext, GF2(bob_reconciled.bits), pa_seed, "bob")
    bob_final = injector.maybe_flip_hash_output_bit(bob_final, "bob")

    keys_match = np.array_equal(alice_final, bob_final)
    return keys_match, injector.summary()


# --- Use key0's real, already-reconciled data ---
n = alice_key.get_nr_bits()
Q = real_qber

ell_finite, _, _ = finite_key_output_length(n, Q)
ell_asymptotic = asymptotic_key_length(n, Q)
ell_placeholder = ToeplitzHashing.calculate_length(
    extractor_type="quantum", input_length=n, relative_source_entropy=0.5, error_bound=1e-6,
)

print(f"Comparing output lengths: asymptotic={ell_asymptotic}, "
      f"placeholder={ell_placeholder}, finite-key={ell_finite}")

# --- Assume bob_key already reconciles cleanly against alice_key (fault-free Cascade) ---
# If not already reconciled in this session, reconcile first:
from qne.cascade import ORIGINAL, Reconciliation, MockClassicalSession
session = MockClassicalSession(correct_key=alice_key)
reconciliation = Reconciliation(algorithm=ORIGINAL, classical_session=session,
                                  noisy_key=bob_key, estimated_bit_error_rate=Q, seed=42)
bob_reconciled = reconciliation.reconcile()
assert alice_key.nr_bits_different(bob_reconciled) == 0, "Reconciliation didn't converge cleanly!"

# Fast version of the length-comparison sweep, using
# fast_toeplitz_extract_with_fault (no to_matrix() call at all).
rows = []
for length_type, out_len in [("asymptotic", ell_asymptotic),
                               ("placeholder", ell_placeholder),
                               ("finite_key", ell_finite)]:
    ext = ToeplitzHashing(input_length=alice_key.get_nr_bits(), output_length=out_len)

    for prob in [0.1, 0.3, 0.5]:
        for run in range(10):
            seed = 42 + run
            pa_seed = GF2.Random(ext.seed_length)
            injector = SDCFaultInjector(toeplitz_matrix_prob=prob, seed=seed + 2)

            alice_final = injector.fast_toeplitz_extract_with_fault(
                ext, GF2(alice_key.bits), pa_seed, "alice")
            bob_final = injector.fast_toeplitz_extract_with_fault(
                ext, GF2(bob_reconciled.bits), pa_seed, "bob")

            keys_match = np.array_equal(alice_final, bob_final)
            rows.append({
                "length_type": length_type, "out_len": out_len, "toeplitz_prob": prob,
                "run": run, "keys_match": keys_match, "faults_fired": injector.summary(),
            })

df_lengthcompare = pd.DataFrame(rows)
df_lengthcompare["mismatch"] = ~df_lengthcompare["keys_match"]
df_lengthcompare.to_csv(str(PROJECT_DIR / "results" / "sdc_lengthtype_comparison.csv"), index=False)

print(df_lengthcompare[["length_type", "toeplitz_prob", "run", "faults_fired", "mismatch"]].to_string())

Comparing output lengths: asymptotic=3181, placeholder=1900, finite-key=2282
    length_type  toeplitz_prob  run            faults_fired  mismatch
0    asymptotic            0.1    0                      {}     False
1    asymptotic            0.1    1                      {}     False
2    asymptotic            0.1    2  {'toeplitz_matrix': 1}      True
3    asymptotic            0.1    3                      {}     False
4    asymptotic            0.1    4                      {}     False
5    asymptotic            0.1    5                      {}     False
6    asymptotic            0.1    6                      {}     False
7    asymptotic            0.1    7                      {}     False
8    asymptotic            0.1    8                      {}     False
9    asymptotic            0.1    9  {'toeplitz_matrix': 1}      True
10   asymptotic            0.3    0  {'toeplitz_matrix': 1}     False
11   asymptotic            0.3    1                      {}     False
12   asymptot

In [6]:
"""
Side-by-side comparison of mismatch rate across the three output-length
types, with a statistical test for whether any real difference exists.
"""
import pandas as pd
from scipy import stats

df_lengthcompare = pd.read_csv(str(PROJECT_DIR / "results" / "sdc_lengthtype_comparison.csv"))

# --- Side-by-side summary table ---
summary = df_lengthcompare.groupby(["length_type", "toeplitz_prob"])["mismatch"].agg(
    ['sum', 'count', 'mean']).rename(columns={'sum': 'n_mismatch', 'count': 'n_total', 'mean': 'mismatch_rate'})
print("=== Mismatch rate by length type and probability ===")
print(summary.to_string())

# Pivot for a clean side-by-side view
pivot = df_lengthcompare.pivot_table(index="toeplitz_prob", columns="length_type",
                                        values="mismatch", aggfunc="mean")
print("\n=== Side-by-side (rows=probability, columns=length type) ===")
print(pivot.to_string())


# ============================================================
# Statistical test: is there a real difference between length types?
# ============================================================
# Since faults_fired is identical across length types for matching
# (prob, run) pairs (confirmed from your data -- same seed => same
# firing pattern), a paired test is appropriate here, not an
# independent-samples test.

print("\n=== McNemar's test: paired comparison of mismatch outcomes ===")
# McNemar's test is the correct choice for paired binary outcomes
# (same trials, same seeds, just different length -- not independent samples)

for prob in df_lengthcompare["toeplitz_prob"].unique():
    sub = df_lengthcompare[df_lengthcompare["toeplitz_prob"] == prob]

    asym = sub[sub["length_type"] == "asymptotic"].sort_values("run")["mismatch"].values
    fkey = sub[sub["length_type"] == "finite_key"].sort_values("run")["mismatch"].values
    plch = sub[sub["length_type"] == "placeholder"].sort_values("run")["mismatch"].values

    # Build 2x2 contingency table: asymptotic vs finite_key
    both_true = ((asym == True) & (fkey == True)).sum()
    only_asym = ((asym == True) & (fkey == False)).sum()
    only_fkey = ((asym == False) & (fkey == True)).sum()
    both_false = ((asym == False) & (fkey == False)).sum()

    table = [[both_true, only_asym], [only_fkey, both_false]]

    # McNemar's test focuses on the discordant pairs (only_asym, only_fkey)
    n_discordant = only_asym + only_fkey
    if n_discordant == 0:
        print(f"prob={prob}: no discordant pairs (asymptotic and finite_key agree on every trial) -- no difference detected")
        continue

    # scipy doesn't expose a direct McNemar's test; use the classic
    # binomial-based version on the discordant pairs instead.
    from scipy.stats import binomtest
    p_value = binomtest(min(only_asym, only_fkey), n_discordant, 0.5).pvalue if n_discordant > 0 else 1.0

    print(f"prob={prob}: discordant pairs = {n_discordant} (asym-only={only_asym}, fkey-only={only_fkey}), "
          f"McNemar p-value = {p_value:.4f}")

=== Mismatch rate by length type and probability ===
                           n_mismatch  n_total  mismatch_rate
length_type toeplitz_prob                                    
asymptotic  0.1                     2       10            0.2
            0.3                     2       10            0.2
            0.5                     4       10            0.4
finite_key  0.1                     2       10            0.2
            0.3                     2       10            0.2
            0.5                     4       10            0.4
placeholder 0.1                     2       10            0.2
            0.3                     2       10            0.2
            0.5                     4       10            0.4

=== Side-by-side (rows=probability, columns=length type) ===
length_type    asymptotic  finite_key  placeholder
toeplitz_prob                                     
0.1                   0.2         0.2          0.2
0.3                   0.2         0.2          0.2

In [7]:
"""
Extend the length-type comparison to hash-output faults (should be
trivially unaffected by output length) and reconciliation-state faults
(structurally can't be affected, since reconciliation happens before
the length choice is applied) -- for completeness and explicit
empirical confirmation.
"""
import pandas as pd
import numpy as np

# ============================================================
# Hash-output faults across the three length types
# ============================================================
rows_hash = []
for length_type, out_len in [("asymptotic", ell_asymptotic),
                               ("placeholder", ell_placeholder),
                               ("finite_key", ell_finite)]:
    ext = ToeplitzHashing(input_length=alice_key.get_nr_bits(), output_length=out_len)

    for prob in [0.1, 0.3, 0.5]:
        for run in range(10):
            seed = 42 + run
            pa_seed = GF2.Random(ext.seed_length)
            injector = SDCFaultInjector(hash_output_prob=prob, seed=seed + 2)

            alice_final = np.array(ext.extract(GF2(alice_key.bits), pa_seed)).copy()
            alice_final = injector.maybe_flip_hash_output_bit(alice_final, "alice")

            bob_final = np.array(ext.extract(GF2(bob_reconciled.bits), pa_seed)).copy()
            bob_final = injector.maybe_flip_hash_output_bit(bob_final, "bob")

            keys_match = np.array_equal(alice_final, bob_final)
            rows_hash.append({
                "length_type": length_type, "out_len": out_len, "hash_prob": prob,
                "run": run, "keys_match": keys_match, "faults_fired": injector.summary(),
            })

df_hash_compare = pd.DataFrame(rows_hash)
df_hash_compare["mismatch"] = ~df_hash_compare["keys_match"]
df_hash_compare.to_csv(str(PROJECT_DIR / "results" / "sdc_lengthtype_comparison_hash.csv"), index=False)

pivot_hash = df_hash_compare.pivot_table(index="hash_prob", columns="length_type",
                                            values="mismatch", aggfunc="mean")
print("=== Hash faults: mismatch rate by length type ===")
print(pivot_hash.to_string())


# ============================================================
# Reconciliation-state faults across the three length types
# (reconciliation runs BEFORE the length choice applies -- testing
# this confirms the length choice has zero effect, as expected)
# ============================================================
from qne.cascade import ORIGINAL, Reconciliation, MockClassicalSession

rows_recon = []
for length_type, out_len in [("asymptotic", ell_asymptotic),
                               ("placeholder", ell_placeholder),
                               ("finite_key", ell_finite)]:
    for prob in [0.03, 0.1, 0.3]:
        for run in range(10):
            seed = 42 + run
            injector = SDCFaultInjector(reconciliation_state_prob=prob, seed=seed + 2)

            session = MockClassicalSession(correct_key=alice_key)
            reconciliation = Reconciliation(
                algorithm=ORIGINAL, classical_session=session, noisy_key=bob_key,
                estimated_bit_error_rate=real_qber, seed=seed, fault_injector=injector,
            )
            try:
                bob_reconciled_test = reconciliation.reconcile()
                non_convergent = False
                remaining = alice_key.nr_bits_different(bob_reconciled_test)
            except RuntimeError:
                non_convergent = True
                remaining = None

            rows_recon.append({
                "length_type": length_type, "out_len": out_len, "reconciliation_prob": prob,
                "run": run, "non_convergent": non_convergent, "remaining_errors": remaining,
                "faults_fired": injector.summary(),
            })

df_recon_compare = pd.DataFrame(rows_recon)
df_recon_compare.to_csv(str(PROJECT_DIR / "results" / "sdc_lengthtype_comparison_reconciliation.csv"), index=False)

pivot_recon = df_recon_compare.pivot_table(index="reconciliation_prob", columns="length_type",
                                              values="non_convergent", aggfunc="mean")
print("\n=== Reconciliation faults: non-convergence rate by length type ===")
print(pivot_recon.to_string())

# Explicit check: are outcomes literally identical across length types (as predicted)?
identical_hash = df_hash_compare.pivot_table(index=["hash_prob", "run"], columns="length_type",
                                                values="mismatch").nunique(axis=1).eq(1).all()
identical_recon = df_recon_compare.pivot_table(index=["reconciliation_prob", "run"], columns="length_type",
                                                  values="non_convergent").nunique(axis=1).eq(1).all()
print(f"\nHash outcomes identical across all length types: {identical_hash}")
print(f"Reconciliation outcomes identical across all length types: {identical_recon}")

=== Hash faults: mismatch rate by length type ===
length_type  asymptotic  finite_key  placeholder
hash_prob                                       
0.1                 0.2         0.2          0.2
0.3                 0.3         0.3          0.3
0.5                 0.6         0.6          0.6

=== Reconciliation faults: non-convergence rate by length type ===
length_type          asymptotic  finite_key  placeholder
reconciliation_prob                                     
0.03                        0.0         0.0          0.0
0.10                        0.0         0.0          0.0
0.30                        0.0         0.0          0.0

Hash outcomes identical across all length types: True
Reconciliation outcomes identical across all length types: True


In [9]:
df_fk = pd.read_csv(str(PROJECT_DIR / "results" / "finite_key_analysis_all_keys.csv"))
df_fk["gap_pct"] = df_fk["reduction_vs_asymptotic_pct"]
print(df_fk[["key_index", "qber", "gap_pct"]].sort_values("qber"))

  key_index      qber    gap_pct
1         1  0.009489  27.916407
3         3  0.009699  27.917834
4         4  0.010540  27.930715
0         0  0.011024  28.380528
5  original  0.011352  28.261553
2         2  0.013826  29.538565


In [10]:
df_fk = pd.read_csv(str(PROJECT_DIR / "results" / "finite_key_analysis_all_keys.csv"))
print(df_fk[["key_index", "n_bits", "qber", "ell_asymptotic", "ell_finite_key", "reduction_vs_asymptotic_pct"]])

  key_index  n_bits      qber  ell_asymptotic  ell_finite_key  \
0         0    3810  0.011024            3143            2251   
1         1    3794  0.009489            3206            2311   
2         2    3761  0.013826            2969            2092   
3         3    3815  0.009699            3213            2316   
4         4    3890  0.010540            3233            2330   
5  original    3876  0.011352            3181            2282   

   reduction_vs_asymptotic_pct  
0                    28.380528  
1                    27.916407  
2                    29.538565  
3                    27.917834  
4                    27.930715  
5                    28.261553  


In [10]:
import pandas as pd
import glob

def audit_realchannel_csv(path):
    """
    Audit a saved real-channel sweep CSV for rows where keys_match=False
    might actually have been a measurement failure under pre-fix code,
    rather than a confirmed key mismatch.
    """
    df = pd.read_csv(path)
    print(f"\n=== {path} ({len(df)} rows) ===")

    if "keys_match" not in df.columns:
        print("  No 'keys_match' column found -- nothing to audit here.")
        return df

    available_cols = set(df.columns)
    print(f"  Columns available: {sorted(available_cols)}")

    failure_signal_cols = [
        "alice_output_path", "bob_output_path", "error",
        "raw_stdout", "raw_stderr", "alice_measurement_error", "alice_error",
    ]
    present_signals = [c for c in failure_signal_cols if c in available_cols]
    print(f"  Failure-signal columns present: {present_signals or 'NONE'}")

    def is_missing(val):
        return pd.isna(val) or val in (None, "", "None", "nan")

    def classify(row):
        km = row.get("keys_match")
        # Treat old-code False the same whether stored as bool False or
        # string "False" (CSV round-tripping sometimes does this)
        if km not in (False, "False", 0):
            return "not_applicable"

        if "alice_output_path" in available_cols and is_missing(row.get("alice_output_path")):
            return "likely_measurement_failure"

        if "error" in available_cols and not is_missing(row.get("error")):
            return "likely_measurement_failure"

        if "raw_stderr" in available_cols and not is_missing(row.get("raw_stderr")):
            return "likely_measurement_failure"

        if "alice_measurement_error" in available_cols and not is_missing(row.get("alice_measurement_error")):
            return "likely_measurement_failure"

        if "alice_error" in available_cols and not is_missing(row.get("alice_error")):
            return "likely_measurement_failure"

        if not present_signals:
            return "unauditable"

        return "confirmed_mismatch"

    df["audit_status"] = df.apply(classify, axis=1)

    counts = df["audit_status"].value_counts()
    print(f"  Audit results:\n{counts.to_string()}")

    n_needs_attention = counts.get("unauditable", 0) + counts.get("likely_measurement_failure", 0)
    if n_needs_attention > 0:
        print(f"  >>> {n_needs_attention} row(s) need attention:")
        print(f"      - 'likely_measurement_failure': relabel as unmeasurable, don't count as SDC detection")
        print(f"      - 'unauditable': no saved evidence either way -- consider rerunning")
    else:
        print("  All keys_match=False rows appear to be confirmed mismatches.")

    return df


def audit_all_files(pattern="/home/fabric/work/qkd-dependability/results/sdc_realchannel_*.csv"):
    files = sorted(glob.glob(pattern))
    if not files:
        print(f"No files matched pattern: {pattern}")
        return pd.DataFrame()

    summary_rows = []
    for path in files:
        df = audit_realchannel_csv(path)
        if "audit_status" not in df.columns:
            continue
        counts = df["audit_status"].value_counts().to_dict()
        summary_rows.append({"file": path, "n_rows": len(df), **counts})

    summary = pd.DataFrame(summary_rows).fillna(0)
    print("\n=== Combined summary across all files ===")
    print(summary.to_string(index=False))
    return summary


# --- run it ---
summary = audit_all_files("/home/fabric/work/qkd-dependability/results/sdc_realchannel_*.csv")

# --- inspect the flagged rows in a specific file ---
df = audit_realchannel_csv("/home/fabric/work/qkd-dependability/results/sdc_realchannel_toeplitz_n20.csv")
needs_attention = df[df["audit_status"].isin(["unauditable", "likely_measurement_failure"])]
print(needs_attention)


=== /home/fabric/work/qkd-dependability/results/sdc_realchannel_hash_n20.csv (160 rows) ===
  Columns available: ['alice_output_path', 'bob_output_path', 'elapsed_seconds', 'error', 'fault_type', 'faults_fired', 'hash_prob', 'keys_match', 'non_convergent', 'output_path', 'prob', 'reconciliation_prob', 'remaining_errors_after_reconciliation', 'run', 'secure_key_length', 'seed', 'toeplitz_prob', 'total_corrections']
  Failure-signal columns present: ['alice_output_path', 'bob_output_path', 'error']
  Audit results:
audit_status
not_applicable        139
confirmed_mismatch     21
  All keys_match=False rows appear to be confirmed mismatches.

=== /home/fabric/work/qkd-dependability/results/sdc_realchannel_reconciliation_n20.csv (120 rows) ===
  No 'keys_match' column found -- nothing to audit here.

=== /home/fabric/work/qkd-dependability/results/sdc_realchannel_toeplitz_n20.csv (160 rows) ===
  Columns available: ['alice_output_path', 'bob_output_path', 'elapsed_seconds', 'error', 'faul

In [11]:
import pandas as pd

for path in [
    "/home/fabric/work/qkd-dependability/results/sdc_realchannel_hash_n20.csv",
    "/home/fabric/work/qkd-dependability/results/sdc_realchannel_toeplitz_n20.csv",
    "/home/fabric/work/qkd-dependability/results/sdc_realchannel_toeplitz_n20_partial.csv",
]:
    df = pd.read_csv(path)
    mismatched = df[df["keys_match"] == False]
    print(f"\n=== {path} ===")
    print(f"{len(mismatched)} mismatched rows")
    # Check for any secondary evidence the comparison was real:
    # secure_key_length > 0 and non_convergent == False strongly suggests
    # PA actually ran and produced two distinct real keys to compare
    suspicious = mismatched[(mismatched["non_convergent"] == True) |
                             (mismatched["secure_key_length"].isna()) |
                             (mismatched["secure_key_length"] == 0)]
    print(f"{len(suspicious)} of those have non_convergent=True or "
          f"secure_key_length null/0 -- these should NOT be counted as "
          f"confirmed SDC mismatches, since PA may not have completed on one side")
    if len(suspicious) > 0:
        print(suspicious[["run", "seed", "non_convergent", "secure_key_length",
                           "alice_output_path", "bob_output_path"]])


=== /home/fabric/work/qkd-dependability/results/sdc_realchannel_hash_n20.csv ===
21 mismatched rows
0 of those have non_convergent=True or secure_key_length null/0 -- these should NOT be counted as confirmed SDC mismatches, since PA may not have completed on one side

=== /home/fabric/work/qkd-dependability/results/sdc_realchannel_toeplitz_n20.csv ===
11 mismatched rows
0 of those have non_convergent=True or secure_key_length null/0 -- these should NOT be counted as confirmed SDC mismatches, since PA may not have completed on one side

=== /home/fabric/work/qkd-dependability/results/sdc_realchannel_toeplitz_n20_partial.csv ===
11 mismatched rows
0 of those have non_convergent=True or secure_key_length null/0 -- these should NOT be counted as confirmed SDC mismatches, since PA may not have completed on one side


In [12]:
from qne.cascade.finite_key import optimize_nu

for m in [1000, 2000, 5000, 10000]:  # replace with your real observed m values
    k = int(np.ceil(np.sqrt(m)))
    n = m - k
    try:
        best_nu, ell, r, t = optimize_nu(n, k, Q=0.02)  # use your real observed QBER
        print(f"m={m}, k={k} (sqrt), n={n} -> ell={ell:.0f}")
    except ValueError as e:
        print(f"m={m}, k={k} (sqrt) -> INFEASIBLE: {e}")

ImportError: cannot import name 'optimize_nu' from 'qne.cascade.finite_key' (/home/fabric/work/qkd-dependability/qne/cascade/finite_key.py)